In [ ]:
%load_ext autoreload
%autoreload 2

# Updating Ligands and Proteins

This notebook demonstrates how to **update** existing ligand and protein records on the data platform.

- **`sync()`** links to an existing record by identity (SMILES / file path) or creates a new one.
- **`update()`** patches a record you already have an ID for (e.g. after editing a structure file).

Updates are **immutable-versioned**: the platform closes the current row and inserts a new version. The **ID stays the same**; `version` increments.

## Setup

Credentials come from **`~/.deeporigin/`** (run `deeporigin login` once). We use `DeepOriginClient.from_disk()` so a repo `.env` or `DO_*` variables in the kernel do not override disk config.

In [ ]:
import uuid

from deeporigin.drug_discovery import BRD_DATA_DIR, Ligand, Protein
from deeporigin.platform import DeepOriginClient

# Disk config only (~/.deeporigin/ after `deeporigin login`).
# Do not use DeepOriginClient() here — it prefers DO_* env vars over disk.
client = DeepOriginClient.from_disk("dev")
client

## Entities layer — update ligand metadata and mol_file

Use `client.entities.update_ligand()` to patch mutable fields on an existing ID.

In [ ]:
tag = f"notebook-{uuid.uuid4().hex[:12]}"

created = client.entities.create_ligand(
    smiles="CC(C)O",
    name=f"entity-updates-demo-{tag}",
    variant_name_tag=tag,
)
ligand_id = created["data"]["id"]
version_before = created["data"]["version"]
print(f"Created ligand {ligand_id} at version {version_before}")

updated = client.entities.update_ligand(
    ligand_id,
    mol_file=f"entities/ligands/demo-{tag}.sdf",
    name="Renamed in notebook",
)
row = updated["data"][0]
print(f"Updated to version {row['version']}, mol_file={row['mol_file']!r}")

## Entities layer — update protein file_path

In [ ]:
remote_pdb = f"entities/proteins/demo-{uuid.uuid4().hex[:8]}.pdb"
client.files.upload(BRD_DATA_DIR / "brd.pdb", remote_pdb)

created_protein = client.entities.create_protein(file_path=remote_pdb)
protein_id = created_protein["data"]["id"]

new_path = f"entities/proteins/updated-{uuid.uuid4().hex[:8]}.pdb"
client.files.upload(BRD_DATA_DIR / "brd.pdb", new_path)


patched = client.entities.update_protein(protein_id, file_path=new_path)
protein_row = patched["data"][0]
print(f"Protein {protein_id} now at version {protein_row['version']}")
print(f"file_path={protein_row['file_path']!r}")

In [ ]:
Protein.from_id("0APX8MPXT8RJ6", client=client)

## Batch update

Patch multiple ligands in one request with `client.entities.batch_update()`.

In [ ]:
batch_ids: list[str] = []
for i in range(2):
    t = f"batch-{uuid.uuid4().hex[:10]}-{i}"
    resp = client.entities.create_ligand(smiles=f"C{'C' * i}O", variant_name_tag=t)
    batch_ids.append(resp["data"]["id"])

batch_result = client.entities.batch_update(
    "ligands",
    updates=[
        {"id": batch_ids[0], "set": {"name": "batch-alpha"}},
        {"id": batch_ids[1], "set": {"name": "batch-beta"}},
    ],
    returning=["id", "name", "version"],
)
for row in batch_result["data"]:
    print(row["id"], row["name"], "v" + str(row["version"]))

## Domain layer — `Ligand.update()` and `Protein.update()`

When you already have a platform ID and a local structure file, domain objects can upload and patch in one call.

In [ ]:
# Register a ligand, then patch mol_file on the same ID
ligand = Ligand.from_sdf(BRD_DATA_DIR / "brd-2.sdf")
ligand.register(client=client)
ligand_id = ligand.id
print(f"Registered {ligand_id}, mol_file={ligand.remote_path!r}")

# Simulate editing the structure locally — load a different SDF, keep the ID
edited = Ligand.from_sdf(BRD_DATA_DIR / "brd-3.sdf")
edited.id = ligand_id
edited.update(client=client)
print(f"After update: mol_file={edited.remote_path!r}")

protein = Protein.from_file(BRD_DATA_DIR / "brd.pdb")
protein.sync(client=client)
print(f"Protein id={protein.id}")

new_remote = f"entities/proteins/notebook-{uuid.uuid4().hex[:8]}.pdb"
protein.update(client=client, remote_path=new_remote)
print(f"After update: file_path={protein.remote_path!r}")

## sync vs update

| Method | When to use |
|--------|-------------|
| `sync()` | First-time link-or-create by identity (SMILES / file path). Does **not** change `mol_file` / `file_path` on an existing ID. |
| `update()` | You already have a platform ID and want to upload a new structure file (or pass an explicit remote path). |

If you edit a structure locally and call `sync()` on a ligand that already has an ID, the platform record is **not** updated — call `update()` instead.